In [ ]:
import torch
import torch.nn.functional as F
import numpy as np

import snntorch.spikegen as spikegen
import tonic

from src.temporal_score_cam import TemporalScoreCAM
from src.utils.config_utils import setup_config_dictionary
from src.utils.save_utils import interactive_load_model
from src.utils.plot_utils import plot_frames, plot_cams, plot_combined_frames, combine_polarities

from src.utils.data_utils import get_dataloaders, get_datasets
from src.utils.model_utils import get_model

config = setup_config_dictionary()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(torch.seed())
# cute kitty seed for tiny_imagenet: 11122600975969166168
# torch.manual_seed(4762538657146009718)

# some overwrites allow us to train and infer at the same time without having to change the config file
config["MODEL_NAME"]="basic_snn"  # Options: basic_snn or vgg11
config["ENABLE_BNTT"]= True
config["DATASET_NAME"] ="dvs_gesture"

print("Run this one cell at a time, otherwise interactive loading will not work.")

In [ ]:
gesture_map = {
    0: "hand_clapping",
    1: "right_hand_wave",
    2: "left_hand_wave",
    3: "right_hand_clockwise",
    4: "right_hand_counter_clockwise",
    5: "left_hand_clockwise",
    6: "left_hand_counter_clockwise",
    7: "forearm_roll_forward",
    8: "forearm_roll_backward",
    9: "drums",
    10: "guitar",
    #11: "random_other_gestures", # -> not implemented here
}

no_time_datasets = ["tiny_imagenet"]


if config["DATASET_NAME"] in no_time_datasets:
    train_ds, test_ds = get_datasets(config["DATASET_NAME"])
    # all of them except the last transform
    train_ds.transform.transforms = train_ds.transform.transforms[:-1]
    test_ds.transform.transforms = test_ds.transform.transforms[:-1]
    train_loader = torch.utils.data.DataLoader(
        train_ds, 
        batch_size=1, 
        shuffle=True, 
        collate_fn=tonic.collation.PadTensors(), 
    )

    test_loader = torch.utils.data.DataLoader(
        test_ds, 
        batch_size=1, 
        shuffle=True, 
        collate_fn=tonic.collation.PadTensors(), 
    )
# Need original images/videos for non-neuromorphic datasets as well.
# load the model
else:
    train_loader, test_loader = get_dataloaders(
        config["DATASET_NAME"], 
        1, 
        False,
    )

event_batches, _ = next(iter(train_loader))
print(f"Event batch shape: {event_batches.shape}")

if event_batches.ndim == 5: # time already added
    num_channels = event_batches.shape[2]
elif event_batches.ndim == 4: # time not added yet
    num_channels = event_batches.shape[1]

model = get_model(
        config["MODEL_NAME"],
        num_classes=len(train_loader.dataset.classes),
        num_channels=num_channels,
        time_steps=config["N_TIME_BINS"],
        beta=config["BETA"],
        enable_bntt=config["ENABLE_BNTT"],
    ).to(device)

interactive_load_model(model, config["MODEL_SAVE_DIR"], config["DATASET_NAME"])

In [ ]:
torch.cuda.synchronize()
torch.cuda.empty_cache()

# Visualize using our ScoreCAM implementation
score_cam = TemporalScoreCAM(model)
conv_layers = [module for module in model.modules() if isinstance(module, torch.nn.Conv2d)]
print("Total convolutional layers:", len(conv_layers))

model.eval()
with torch.no_grad():
    for batch_idx, (event_batches, labels) in enumerate(test_loader):
        frames = event_batches[0].cpu().numpy()
        if config["DATASET_NAME"] in no_time_datasets:
            event_batches = spikegen.rate(event_batches, config["N_TIME_BINS"])
            event_batches = event_batches.transpose(0, 1) # from T, B, C, H, W to B, T, C, H, W

        event_batches = event_batches.to(device)
        labels = labels.to(device)
        print(event_batches.shape)
        outputs = model(event_batches) # output shape B, T, C
        peak_class_prediction = torch.max(outputs, dim=1) # time max membrane
        output_class = F.softmax(peak_class_prediction.values, dim=1).cpu().numpy()

        print(f"Output probabilities: {[f'{p:.2%}' for p in output_class[0]]}\n")

        sorted_indices = output_class[0].argsort()[::-1]  # Descending order
        predicted_label = int(sorted_indices[0])
        second_label = int(sorted_indices[1])
        correct_prediction = (predicted_label == labels.item())
        if correct_prediction:
            print(f"Correct prediction with label {predicted_label} with likelihood of {output_class[0][predicted_label]}")
        else:
            print(f"Predicted label {predicted_label} with likelihood of {output_class[0][predicted_label]:.2%}, real label: {labels.item()} with likelihood of {output_class[0][labels.item()]:.2%}")

        static_image = False
        if frames.ndim == 3:
            static_image = True
            frames = np.expand_dims(frames, axis=0)  # Add batch dimension if missing
            frames = np.repeat(frames, axis=0, repeats=config["N_TIME_BINS"])  # Repeat frames for each time step if it's a static image
            
        frames = frames.transpose(0, 2, 3, 1)  # Change to (H, W, C) format

        print("Original Frames:")
        if static_image:
            plot_frames(frames[0:1])
        else:
            plot_frames(frames)

        if num_channels == 2:
            print("Combined Frames (for two polarities):")
            frames = combine_polarities(frames)
            plot_combined_frames(frames)
        step = -(len(conv_layers) // 4) if len(conv_layers) >= 4 else -1
        print("Per-layer CAMs, from later to earlier convolutional layer:")
        for layer_idx in range(len(conv_layers) - 1, 0, step):
            target_layer = conv_layers[layer_idx]
            cam = score_cam.generate_cam(event_batches, target_layer=target_layer)
            if static_image:
                cams_sum = cam.sum(axis=1) # B, NUM_CLASSES, H, W
                batch_size, num_classes, _, _ = cams_sum.shape
                # print(cams_sum.shape)
                # normalize squished cam per class
                flat = cams_sum.view(batch_size, num_classes, -1)
                mins = flat.min(2, keepdim=True)[0].view(batch_size, num_classes, 1, 1)
                maxs = flat.max(2, keepdim=True)[0].view(batch_size, num_classes, 1, 1)
                cams_sum = (cams_sum - mins) / (maxs - mins + 1e-8)
                squished_cam = np.expand_dims(cams_sum, axis=1)
            print(f"Layer {layer_idx+1}/{len(conv_layers) + 1}:")
            print(f"Cams for predicted class: {predicted_label}")
            plot_cams(cam, class_idx=predicted_label, frames=frames)
            print(f"Most important 3 time steps for the predicted class: {score_cam.find_most_important_cam_frames(cam, class_idx=predicted_label)[0][:3]}")
            if static_image:
                print("Cumulative CAMs for predicted class: (Squished across the time dimension for originally static images)")
                plot_cams(squished_cam, class_idx=predicted_label, frames=frames)
            if correct_prediction:
                print(f"Cams for second most likely class: {second_label}")
                plot_cams(cam, class_idx=second_label, frames=frames)
                if static_image:
                    print("Cumulative CAMs for second most likely class: (Squished across the time dimension for originally static images)")
                    plot_cams(squished_cam, class_idx=second_label, frames=frames)
                    print(f"Most important 3 time steps for the second most likely class: {score_cam.find_most_important_cam_frames(cam, class_idx=second_label)[0][:3]}")
            else: # incorrect prediction
                print(f"Cams for real class: {labels.item()}")
                plot_cams(cam, class_idx=labels.item(), frames=frames)
                print(f"Most important 3 time steps for the real class: {score_cam.find_most_important_cam_frames(cam, class_idx=labels.item())[0][:3]}")
                if static_image:
                    print("Cumulative CAMs for real class: (Squished across the time dimension for originally static images)")
                    plot_cams(squished_cam, class_idx=labels.item(), frames=frames)
            break
        break


In [ ]:
import pandas as pd

# Create a DataFrame with membrane potentials
membrane_data = []
for time_step in range(outputs.shape[1]):
    membrane_data.append(outputs[0, time_step, :].cpu().numpy())

# Convert to DataFrame
df = pd.DataFrame(membrane_data, 
                  columns=[f'Neuron_{i}' for i in range(outputs.shape[2])],
                  index=[f'Time_step_{i}' for i in range(outputs.shape[1])])

# Add a totals row
df.loc['Total'] = df.sum(axis=0)

print("Membrane potentials for each time step:")
diff_df = df.iloc[:-1].diff()
diff_df.loc['Total'] = [None] * len(df.columns)
def color_changes(val):
    if pd.isna(val):
        return ''

    abs_change = abs(val)
    max_change = diff_df.dropna().abs().max().max()
    
    if max_change == 0:
        return ''
    
    intensity = min(abs_change / max_change, 1.0)

    alpha = 0.2 + (intensity * 0.6)
    if val > 0:
        return f'background-color: rgba(0, 255, 0, {alpha})'
    elif val < 0:
        return f'background-color: rgba(255, 0, 0, {alpha})'
    else:
        return ''

styled_df = df.style.apply(lambda x: diff_df.loc[x.name].apply(color_changes), axis=1)

display(styled_df)

print(f"\nChange statistics:")
print(f"Maximum increase: {diff_df.dropna().max().max():.4f}")

print(f"\nNeuron total statistics:")
neuron_totals = df.loc['Total']
print(f"Maximum neuron total: {neuron_totals.max():.4f}")

print(f"\nTop 5 largest increases in membrane potential:")
# Get all positive changes and sort them
positive_changes = diff_df.dropna().stack()
positive_changes = positive_changes[positive_changes > 0]
top_increases = positive_changes.nlargest(5)

for i, (location, value) in enumerate(top_increases.items(), 1):
    timestep, neuron = location
    print(f"{i}. {value:.4f} - {neuron} at {timestep}")